# Automotive Marketing Content LoRA Studio

Demo academica para generar piezas visuales de marketing automotriz entrenando un LoRA visual con Diffusers.

El flujo sigue la estructura del notebook `LoRA_Logo_Marca_Colab.ipynb`:

- verificar GPU
- preparar imagenes + captions
- entrenar DreamBooth LoRA sobre SDXL
- cargar el LoRA entrenado
- generar una galeria de assets para distintos placements
- comparar SDXL base vs SDXL + LoRA

En esta version no se fine-tunea un LLM. El LLM es opcional y solo puede usarse para mejorar prompts y negative prompts.

Alcance definido:

- se usaran modelos reales de autos cuando el dataset final este listo
- captions y prompts en ingles por defecto
- configuracion conservadora para Colab/Kaggle free tier

## 0. Verificar GPU

La demo esta pensada para Colab o Kaggle con GPU gratuita. SDXL LoRA requiere VRAM; T4 puede servir para una prueba corta con resolucion 512 y pocos steps.

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("GPU detectada:")
    print("\n".join(result.stdout.split("\n")[:15]))
else:
    print("No se detecto GPU. Activa GPU antes de entrenar el LoRA visual.")

## 1. Instalar librerias

Se usa Diffusers desde GitHub para tener el script DreamBooth LoRA SDXL actualizado, siguiendo el enfoque del notebook de referencia.

In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    !pip install -q --upgrade huggingface_hub
    !pip install -q git+https://github.com/huggingface/diffusers.git
    !pip install -q peft transformers accelerate bitsandbytes safetensors pillow matplotlib pandas
    !pip install -q --upgrade torchao

In [ ]:
import importlib
import json
import math
import os
import random
import shutil
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from PIL import Image, ImageDraw

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

for pkg in ["diffusers", "peft", "transformers", "accelerate", "huggingface_hub"]:
    mod = importlib.import_module(pkg)
    print(f"{pkg}: {mod.__version__}")

## 2. Configurar proyecto, marca/modelo y rutas

Edita estas variables cuando tengas las imagenes y descripciones del modelo real. Los valores actuales son placeholders seguros para preparar el flujo.

In [ ]:
CANDIDATE_ROOTS = [
    Path.cwd(),
    Path("/content/dmc-tp3-gen-multimodal"),
    Path("/kaggle/working/dmc-tp3-gen-multimodal"),
]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "data").exists()), Path.cwd())

BRAND = "Ford"  # Cambiar cuando se defina la marca real del dataset.
MODEL_SERIES = "TBD real model"  # Ejemplo futuro: "Mustang Mach-E", "F-150 Lightning", etc.
TRIGGER_WORD = "REALCARMODEL"  # Palabra unica usada en todos los captions y prompts.
INSTANCE_PROMPT = f"{TRIGGER_WORD} real car model automotive marketing campaign asset"

MODEL_NAME = "stabilityai/stable-diffusion-xl-base-1.0"
DATASET_DIR = PROJECT_ROOT / "data" / "car_campaign_lora" / "images"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "automotive-lora"
ASSET_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "generated_assets"
EVAL_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "evaluation"

for path in [DATASET_DIR, OUTPUT_DIR, ASSET_OUTPUT_DIR, EVAL_OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset dir:", DATASET_DIR)
print("LoRA output:", OUTPUT_DIR)
print("Trigger word:", TRIGGER_WORD)

## 3. Cargar imagenes + metadata captions

Coloca imagenes en `data/car_campaign_lora/images/` y registra cada imagen en `data/car_campaign_lora/metadata.csv`.

Ejemplo:

```csv
file_path,caption
./images/real_car_model_01.png,"REALCARMODEL real car model, front three quarter view, metallic blue paint, studio automotive photography, premium lighting"
```

La caption debe estar en ingles y describir trigger word, auto, vista, color, contexto e iluminacion.

Nota: el script DreamBooth LoRA SDXL usado en esta demo recibe un `instance_prompt` comun. El metadata se usa para validar el dataset, documentar captions y alimentar el prompt builder; si luego migramos a un script caption-aware, esta estructura ya queda lista.

In [ ]:
SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}
METADATA_PATH = PROJECT_ROOT / "data" / "car_campaign_lora" / "metadata.csv"
if not METADATA_PATH.exists():
    METADATA_PATH = PROJECT_ROOT / "data" / "car_campaign_lora" / "metadata_template.csv"

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        "No se encontro metadata.csv ni metadata_template.csv en data/car_campaign_lora/. "
        "Crea un CSV con columnas file_path,caption."
    )

metadata_df = pd.read_csv(METADATA_PATH)
required_columns = {"file_path", "caption"}
missing_columns = required_columns - set(metadata_df.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas en metadata: {sorted(missing_columns)}")

records = []
metadata_root = METADATA_PATH.parent

for _, row in metadata_df.iterrows():
    raw_path = Path(str(row["file_path"]))
    image_path = raw_path if raw_path.is_absolute() else (metadata_root / raw_path).resolve()
    caption = str(row["caption"]).strip()
    if image_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise ValueError(f"Extension no soportada para {image_path}")
    if not image_path.exists():
        raise FileNotFoundError(f"No se encontro la imagen declarada en metadata: {image_path}")
    if not caption:
        raise ValueError(f"Caption vacia para {image_path}")
    records.append({"image": image_path, "caption": caption})

if not records:
    raise FileNotFoundError(
        "No se encontraron filas validas en metadata. "
        "Agrega al menos 8-15 imagenes del auto con captions en ingles."
    )

print(f"Metadata usada: {METADATA_PATH}")
print(f"Imagenes declaradas: {len(records)}")
for record in records[:5]:
    print(record["image"].name, "->", record["caption"])
    display(Image.open(record["image"]).resize((256, 256)))

## 4. Reorganizar dataset para DreamBooth LoRA

El script oficial de Diffusers usa una carpeta de imagenes y un `instance_prompt`. Las captions del metadata se mantienen para trazabilidad, QA del dataset y construccion de prompts de campana.

In [ ]:
TRAIN_IMAGES_DIR = PROJECT_ROOT / "outputs" / "automotive_training_images"

if TRAIN_IMAGES_DIR.exists():
    shutil.rmtree(TRAIN_IMAGES_DIR)
TRAIN_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

dataset_summary = []
for idx, record in enumerate(records, start=1):
    src = record["image"]
    dst = TRAIN_IMAGES_DIR / f"{TRIGGER_WORD.lower()}_{idx:03d}.png"
    img = Image.open(src).convert("RGB")
    img.save(dst)
    dataset_summary.append({"file_name": dst.name, "caption": record["caption"]})

summary_df = pd.DataFrame(dataset_summary)
summary_path = EVAL_OUTPUT_DIR / "training_dataset_summary.csv"
summary_df.to_csv(summary_path, index=False)

print(f"Dataset DreamBooth preparado en: {TRAIN_IMAGES_DIR}")
print(f"Resumen guardado en: {summary_path}")
display(summary_df.head())

## 5. Descargar script oficial DreamBooth LoRA SDXL

In [ ]:
SCRIPT_PATH = PROJECT_ROOT / "outputs" / "train_dreambooth_lora_sdxl.py"

if not SCRIPT_PATH.exists():
    !wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_sdxl.py -O {SCRIPT_PATH}

print("Script:", SCRIPT_PATH)
print("Existe:", SCRIPT_PATH.exists())

## 6. Parametros de entrenamiento

Valores pensados para Colab/Kaggle free tier. Para resultado final, subir `MAX_TRAIN_STEPS` a 400-800 si la GPU y el tiempo lo permiten.

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

RESOLUTION = 512
MAX_TRAIN_STEPS = 250  # Free-tier friendly. Subir a 400-800 si la GPU y el tiempo lo permiten.
LORA_RANK = 16
LEARNING_RATE = 1e-4
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
MIXED_PRECISION = "fp16"

print({
    "model": MODEL_NAME,
    "resolution": RESOLUTION,
    "max_train_steps": MAX_TRAIN_STEPS,
    "rank": LORA_RANK,
    "learning_rate": LEARNING_RATE,
    "instance_prompt": INSTANCE_PROMPT,
})

## 7. Entrenar LoRA visual

Esta celda entrena el LoRA sobre las imagenes del auto. Puede tomar de 20 a 60 minutos segun GPU y numero de pasos.

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    cmd = [
        "accelerate", "launch", str(SCRIPT_PATH),
        f"--pretrained_model_name_or_path={MODEL_NAME}",
        f"--instance_data_dir={TRAIN_IMAGES_DIR}",
        f"--output_dir={OUTPUT_DIR}",
        f"--instance_prompt={INSTANCE_PROMPT}",
        f"--resolution={RESOLUTION}",
        f"--train_batch_size={TRAIN_BATCH_SIZE}",
        f"--gradient_accumulation_steps={GRADIENT_ACCUMULATION_STEPS}",
        f"--learning_rate={LEARNING_RATE}",
        "--lr_scheduler=constant",
        "--lr_warmup_steps=0",
        f"--max_train_steps={MAX_TRAIN_STEPS}",
        f"--rank={LORA_RANK}",
        f"--seed={SEED}",
        f"--mixed_precision={MIXED_PRECISION}",
        "--gradient_checkpointing",
    ]
    print("Ejecutando:")
    print(" ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError("El entrenamiento LoRA fallo. Revisa memoria GPU, versiones o rutas.")
else:
    print("RUN_TRAINING=False. Se omitio entrenamiento.")

## 8. Verificar archivos LoRA generados

In [ ]:
lora_files = sorted(OUTPUT_DIR.rglob("*.safetensors"))
if not lora_files:
    raise FileNotFoundError(f"No se encontraron .safetensors en {OUTPUT_DIR}. Ejecuta entrenamiento o carga un LoRA existente.")

for file in lora_files:
    print(file.relative_to(OUTPUT_DIR), f"{file.stat().st_size / 1024**2:.2f} MB")

LORA_PATH = lora_files[-1]
print("LoRA seleccionado:", LORA_PATH)

## 9. Cargar SDXL base y aplicar LoRA

In [ ]:
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    use_safetensors=True,
    variant="fp16" if torch.cuda.is_available() else None,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")

pipe.load_lora_weights(str(OUTPUT_DIR))
print("SDXL + LoRA cargado.")

## 10. Construir prompts por placement

Las piezas se generan con dimensiones diferentes segun canal. Se evita texto legible porque los modelos de difusion suelen deformarlo.

In [ ]:
campaign_brief = {
    "brand": BRAND,
    "model_series": MODEL_SERIES,
    "trigger_word": TRIGGER_WORD,
    "launch_message": "new model launch campaign",
    "target_audience": "professionals 30-45",
    "market": "US and Latin America",
    "tone": "premium, confident, innovative",
}

PLACEMENTS = [
    {"name": "website_hero", "label": "website hero banner", "width": 768, "height": 432, "composition": "wide horizontal 16:9 layout with space for headline overlay"},
    {"name": "instagram_feed", "label": "Instagram feed ad", "width": 640, "height": 640, "composition": "square centered composition"},
    {"name": "vertical_story", "label": "Instagram story or TikTok vertical ad", "width": 432, "height": 768, "composition": "vertical 9:16 mobile-first composition"},
    {"name": "showroom_poster", "label": "dealership showroom poster", "width": 640, "height": 832, "composition": "premium vertical print poster composition"},
    {"name": "highway_billboard", "label": "highway billboard", "width": 896, "height": 384, "composition": "ultra-wide billboard composition readable from distance"},
    {"name": "email_header", "label": "email campaign header", "width": 768, "height": 320, "composition": "clean wide email header composition"},
]

NEGATIVE_PROMPT = (
    "blurry, low quality, watermark, distorted text, malformed logo, extra wheels, "
    "deformed car, broken headlights, bad perspective, jpeg artifacts, people with distorted faces"
)

def build_prompt(brief, placement):
    return (
        f"{brief['trigger_word']} real car model for a {placement['label']}, "
        f"{brief['launch_message']}, targeting {brief['target_audience']} in {brief['market']}, "
        f"{brief['tone']} tone, {placement['composition']}, cinematic automotive photography, "
        "clean reflections, premium lighting, modern city or showroom environment, no readable text, high quality commercial advertising"
    )

prompt_rows = []
for idx, placement in enumerate(PLACEMENTS):
    prompt_rows.append({
        **placement,
        "prompt": build_prompt(campaign_brief, placement),
        "negative_prompt": NEGATIVE_PROMPT,
        "seed": SEED + idx,
    })

prompts_df = pd.DataFrame(prompt_rows)
display(prompts_df[["name", "width", "height", "seed", "prompt"]])

## 11. Opcional: mejorar prompts con LLM

El LLM no se fine-tunea. Esta celda queda apagada por defecto para mantener el flujo reproducible y enfocado en LoRA visual.

In [ ]:
USE_LLM_PROMPT_REFINER = False

if USE_LLM_PROMPT_REFINER:
    from transformers import pipeline

    refiner = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-0.5B-Instruct",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )

    def refine_prompt(prompt):
        instruction = (
            "Improve this image generation prompt for an automotive marketing campaign. "
            "Keep the trigger word exactly and avoid readable text. Return only the improved prompt.\n"
            f"Prompt: {prompt}"
        )
        return refiner(instruction, max_new_tokens=160, do_sample=False)[0]["generated_text"].split("Prompt:")[-1].strip()

    prompts_df["prompt"] = prompts_df["prompt"].map(refine_prompt)
else:
    print("LLM prompt refiner desactivado. Usando prompts de plantilla.")

## 12. Generar assets con LoRA

In [ ]:
def generate_asset(row, use_lora=True):
    if use_lora:
        pipe.load_lora_weights(str(OUTPUT_DIR))
        suffix = "lora"
    else:
        pipe.unload_lora_weights()
        suffix = "base"

    generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(int(row["seed"]))
    start = time.perf_counter()
    image = pipe(
        prompt=row["prompt"],
        negative_prompt=row["negative_prompt"],
        width=int(row["width"]),
        height=int(row["height"]),
        num_inference_steps=30,
        guidance_scale=7.0,
        generator=generator,
    ).images[0]
    latency = time.perf_counter() - start
    image_path = ASSET_OUTPUT_DIR / f"{row['name']}_{suffix}_seed_{row['seed']}.png"
    image.save(image_path)
    return image, image_path, latency

asset_records = []
for _, row in prompts_df.iterrows():
    image, image_path, latency = generate_asset(row, use_lora=True)
    asset_records.append({
        "placement": row["name"],
        "path": str(image_path),
        "prompt": row["prompt"],
        "negative_prompt": row["negative_prompt"],
        "seed": int(row["seed"]),
        "width": int(row["width"]),
        "height": int(row["height"]),
        "model": MODEL_NAME,
        "lora": str(OUTPUT_DIR),
        "latency_sec": latency,
        "qualitative_note": "Revisar consistencia del auto, calidad publicitaria y ajuste al placement.",
    })
    print(row["name"], image_path, f"{latency:.2f}s")
    display(image)

metadata_path = EVAL_OUTPUT_DIR / "image_generation_metadata.json"
with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(asset_records, f, ensure_ascii=False, indent=2)

display(pd.DataFrame(asset_records))
print("Metadata guardada en:", metadata_path)

## 13. Comparacion base vs LoRA

Mismo prompt y misma seed para ver el efecto del LoRA visual.

In [ ]:
comparison_rows = prompts_df.head(3).copy()
comparison_records = []

for _, row in comparison_rows.iterrows():
    base_image, base_path, base_latency = generate_asset(row, use_lora=False)
    lora_image, lora_path, lora_latency = generate_asset(row, use_lora=True)
    comparison_records.append({
        "placement": row["name"],
        "base_path": str(base_path),
        "lora_path": str(lora_path),
        "prompt": row["prompt"],
        "seed": int(row["seed"]),
        "base_latency_sec": base_latency,
        "lora_latency_sec": lora_latency,
    })

    canvas = Image.new("RGB", (base_image.width + lora_image.width, max(base_image.height, lora_image.height)), "white")
    canvas.paste(base_image, (0, 0))
    canvas.paste(lora_image, (base_image.width, 0))
    display(canvas)

comparison_path = EVAL_OUTPUT_DIR / "base_vs_lora_comparison.json"
with comparison_path.open("w", encoding="utf-8") as f:
    json.dump(comparison_records, f, ensure_ascii=False, indent=2)

display(pd.DataFrame(comparison_records))
print("Comparacion guardada en:", comparison_path)

## 14. Conclusiones y valor de negocio

El notebook demuestra un pipeline visual:

1. Imagenes del auto + captions.
2. DreamBooth LoRA sobre SDXL con Diffusers.
3. Prompts por placement de marketing.
4. Generacion de assets para web, social, showroom, print, email y highway.
5. Comparacion base vs LoRA para evidenciar personalizacion visual.

Hipotesis de ROI:

```text
ROI estimado = (horas creativas ahorradas * costo hora equipo creativo * campanas mensuales - costo operativo IA) / costo operativo IA
```

Ejemplo: 12 horas ahorradas por campana * 6 campanas/mes * USD 35/hora = USD 2,520. Si operar IA cuesta USD 300/mes, ROI estimado = 7.4x.

Limitaciones: la calidad depende de las imagenes y captions, el modelo puede deformar texto/logos, y los assets finales requieren revision humana antes de uso comercial.